# Companio: audio distress (scream) detection prototype

**Goal.** Given a short audio clip, estimate the probability that it contains a human scream. The backend would use this score to decide whether to start an SOS (high), ask the user to confirm (medium), or ignore it (low).

**Approach.** Transfer learning. A pretrained audio model (YAMNet, trained on Google's AudioSet) turns each clip into an embedding. A small classifier is trained on those embeddings. This needs far less data and compute than training a neural network from scratch.

**Status: prototype.** This notebook is not yet connected to the Spring Boot backend. The numbers it produces come from public datasets, not real emergencies, and the report should say so.

**What it produces (for the report and slides)**
1. A baseline with no training (YAMNet's built-in "Screaming" scores)
2. A trained classifier compared against that baseline
3. Precision, recall, F1, ROC-AUC, PR-AUC and a confusion matrix on a held-out test set
4. HIGH / MEDIUM / LOW thresholds chosen from validation data
5. A false-positive analysis: which everyday sounds fool the model

## How to run

1. Open this file in Google Colab (colab.research.google.com, then File, Upload notebook).
2. Set the runtime to GPU (Runtime, Change runtime type, T4 GPU). It works on CPU too, only slower.
3. Download the **Human Screaming Detection Dataset** zip from Kaggle (linked in the README of the GitHub repo `whats2000/HumanScreamingDetection`). Check its license and cite it in your report.
4. Run the cells from top to bottom. In section 3 you will load that zip (the upload cell for small zips, or the Google Drive cell for big ones), read the folder listing it prints, and set `SCREAM_DIR` in section 4.

## 1. Setup

In [ ]:
# ---- 1. setup ----
!pip install -q librosa soundfile
import os, zipfile, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf, tensorflow_hub as hub
import librosa, joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, roc_auc_score,
                             average_precision_score, precision_recall_curve,
                             f1_score, precision_score, recall_score)
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); random.seed(SEED)
SR = 16000            # YAMNet expects 16 kHz mono audio
MAX_SECONDS = 10      # only the first 10 s of each clip are used
AUDIO_EXT = ('.wav', '.mp3', '.ogg', '.flac', '.m4a')
print('TensorFlow', tf.__version__)

## 2. Load YAMNet

YAMNet is a pretrained audio-event model (521 sound classes from AudioSet). Two things are used from it: its 1024-number **embedding** for each ~1 second of audio (the features for our classifier), and its own class scores, which include "Screaming" (used for the no-training baseline).

In [ ]:
# ---- 2. yamnet ----
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
class_map = pd.read_csv(yamnet.class_map_path().numpy().decode('utf-8'))
class_names = class_map['display_name'].tolist()
SCREAM_CLASSES = ['Screaming', 'Shout', 'Yell', 'Children shouting']
scream_idx = [class_names.index(c) for c in SCREAM_CLASSES if c in class_names]
print(len(class_names), 'classes. Scream-related ones used for the baseline:', [class_names[i] for i in scream_idx])

## 3. Data

* **Negatives (not scream):** ESC-50, 2000 five-second clips across 50 everyday sound categories (dog, siren, crying baby, laughing, door knock, rain ...). It comes with 5 predefined folds so clips from the same source never sit in both train and test.
* **Positives (scream):** the Kaggle Human Screaming Detection Dataset you download yourself.

In [ ]:
# ---- 3a. esc-50 negatives ----
!wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O esc50.zip
!unzip -q -o esc50.zip
ESC_DIR = 'ESC-50-master'
esc = pd.read_csv(f'{ESC_DIR}/meta/esc50.csv')
print(esc.shape, 'clips,', esc['category'].nunique(), 'categories')

In [ ]:
# ---- 3b. kaggle scream dataset: upload the zip you downloaded ----
# (For a big zip, mount Google Drive instead: from google.colab import drive; drive.mount('/content/drive'))
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('kaggle_scream')
print('Folders that contain audio:')
for root, dirs, fs in os.walk('kaggle_scream'):
    n = sum(f.lower().endswith(AUDIO_EXT) for f in fs)
    if n:
        print(f'  {root}  ->  {n} audio files')

### 3b (alternative). Big zip from Google Drive

Use this cell **instead of** the upload cell above if the zip is large (the Kaggle dataset is about 4 GB). Put the zip in your Google Drive first, set `ZIP_PATH` to its file name, then run the cell. It unzips onto Colab's local disk, which is much faster than reading from Drive.

In [ ]:
# ---- 3b-alt. kaggle zip from Google Drive ----
from google.colab import drive
drive.mount('/content/drive')
ZIP_PATH = '/content/drive/MyDrive/PUT_YOUR_ZIP_FILE_NAME_HERE.zip'
assert os.path.exists(ZIP_PATH), 'ZIP_PATH not found. Check the file name in the Files panel (left side, Drive folder).'
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('kaggle_scream')
print('Folders that contain audio:')
for root, dirs, fs in os.walk('kaggle_scream'):
    n = sum(f.lower().endswith(AUDIO_EXT) for f in fs)
    if n:
        print(f'  {root}  ->  {n} audio files')

## 4. Choose the folders

Read the listing above. Set `SCREAM_DIR` to the folder holding the **scream** clips. If the dataset also has a folder of **non-scream** clips, set `NONSCREAM_DIR` too (otherwise leave it `None` and only ESC-50 provides negatives).

In [ ]:
# ---- 4. paths: EDIT THESE ----
SCREAM_DIR = 'kaggle_scream/PUT_THE_SCREAM_FOLDER_HERE'
NONSCREAM_DIR = None   # e.g. 'kaggle_scream/PUT_THE_NON_SCREAM_FOLDER_HERE'

assert os.path.isdir(SCREAM_DIR), 'SCREAM_DIR does not exist. Copy a folder path from the listing in section 3b.'

MAX_PER_CLASS = 1500   # cap per Kaggle class so feature extraction stays around 20 minutes

def list_audio(folder):
    out = []
    for root, _, fs in os.walk(folder):
        out += [os.path.join(root, f) for f in fs if f.lower().endswith(AUDIO_EXT)]
    return sorted(out)

def cap(paths, name):
    print(f'{name}: {len(paths)} clips found', end='')
    if len(paths) > MAX_PER_CLASS:
        idx = np.random.default_rng(SEED).choice(len(paths), MAX_PER_CLASS, replace=False)
        paths = [paths[i] for i in sorted(idx)]
    print(f', using {len(paths)}')
    return paths

rng = np.random.default_rng(SEED)
def random_split(n):
    # 60/20/20 train/val/test, matching ESC-50's 3/1/1 folds
    r = rng.permutation(n); s = np.empty(n, dtype=object)
    s[r[:int(.6 * n)]] = 'train'
    s[r[int(.6 * n):int(.8 * n)]] = 'val'
    s[r[int(.8 * n):]] = 'test'
    return s

rows = []
pos = cap(list_audio(SCREAM_DIR), 'scream')
for p, s in zip(pos, random_split(len(pos))):
    rows.append(dict(path=p, label=1, category='scream', split=s))
for _, r in esc.iterrows():
    split = 'train' if r['fold'] <= 3 else 'val' if r['fold'] == 4 else 'test'
    rows.append(dict(path=f"{ESC_DIR}/audio/{r['filename']}", label=0, category=r['category'], split=split))
if NONSCREAM_DIR:
    neg = cap(list_audio(NONSCREAM_DIR), 'non-scream')
    for p, s in zip(neg, random_split(len(neg))):
        rows.append(dict(path=p, label=0, category='kaggle_nonscream', split=s))
meta = pd.DataFrame(rows)
print(meta.groupby(['split', 'label']).size().unstack(fill_value=0).rename(columns={0: 'not scream', 1: 'scream'}))

## 5. Extract features

Each clip becomes a 2048-number vector: the mean and the max of YAMNet's embeddings over time. The max helps when a short scream sits inside a longer, quieter clip. The first run takes several minutes; results are cached in `features.npz`, so re-running is instant.

In [ ]:
# ---- 5. feature extraction ----
def load_wave(path):
    wav, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_SECONDS)
    if len(wav) < SR:                      # YAMNet needs at least ~1 s
        wav = np.pad(wav, (0, SR - len(wav)))
    return wav.astype(np.float32)

def extract(path):
    wav = load_wave(path)
    scores, emb, _ = yamnet(tf.constant(wav))
    scores, emb = scores.numpy(), emb.numpy()
    feat = np.concatenate([emb.mean(0), emb.max(0)])
    zero = float(scores[:, scream_idx].max()) if scream_idx else 0.0   # baseline score, no training
    return feat, zero

CACHE = 'features.npz'
if os.path.exists(CACHE) and os.path.exists('meta.csv'):
    d = np.load(CACHE); X, zero = d['X'], d['zero']
    meta = pd.read_csv('meta.csv')
else:
    feats, zs, ok = [], [], []
    for i, p in enumerate(meta['path']):
        try:
            f, z = extract(p); feats.append(f); zs.append(z); ok.append(True)
        except Exception as e:
            print('skipped', p, '-', e); ok.append(False)
        if i % 200 == 0:
            print(f'{i}/{len(meta)}')
    meta = meta[ok].reset_index(drop=True)
    X, zero = np.stack(feats), np.array(zs)
    np.savez(CACHE, X=X, zero=zero); meta.to_csv('meta.csv', index=False)
print('features:', X.shape)

## 6. Splits and a no-training baseline

In [ ]:
# ---- 6. splits and baseline ----
y = meta['label'].values
tr, va, te = [(meta['split'] == s).values for s in ('train', 'val', 'test')]

def best_f1_threshold(y_true, s):
    p, r, t = precision_recall_curve(y_true, s)
    f1 = 2 * p * r / np.maximum(p + r, 1e-9)
    return t[np.argmax(f1[:-1])]

def summarize(name, y_true, score, thr):
    pred = (score >= thr).astype(int)
    return dict(model=name,
                precision=precision_score(y_true, pred, zero_division=0),
                recall=recall_score(y_true, pred, zero_division=0),
                f1=f1_score(y_true, pred, zero_division=0),
                roc_auc=roc_auc_score(y_true, score),
                pr_auc=average_precision_score(y_true, score))

t0 = best_f1_threshold(y[va], zero[va])   # threshold picked on validation data only
base = summarize('Baseline: YAMNet built-in scream scores (no training)', y[te], zero[te], t0)
print(f'Baseline threshold (chosen on validation): {t0:.3f}')
pd.DataFrame([base]).round(3)

## 7. Train the classifier

Logistic regression on standardised embeddings, with class weights because scream clips and non-scream clips are unlikely to be equally numerous. The test set is used once, at the end of this section.

In [ ]:
# ---- 7. train and evaluate ----
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(C=0.05, max_iter=3000, class_weight='balanced'))
clf.fit(X[tr], y[tr])
p_val = clf.predict_proba(X[va])[:, 1]
p_te = clf.predict_proba(X[te])[:, 1]

t_f1 = best_f1_threshold(y[va], p_val)
trained = summarize('Trained: YAMNet embeddings + logistic regression', y[te], p_te, t_f1)
print(f'Trained model threshold (max F1 on validation): {t_f1:.3f}\n')
print(classification_report(y[te], (p_te >= t_f1).astype(int), target_names=['not scream', 'scream'], digits=3))
ConfusionMatrixDisplay.from_predictions(y[te], (p_te >= t_f1).astype(int),
                                        display_labels=['not scream', 'scream'], cmap='Blues')
plt.title('Test-set confusion matrix'); plt.show()
pd.DataFrame([base, trained]).round(3)

## 8. HIGH / MEDIUM / LOW bands

In the app, a **HIGH** score would start an SOS straight away, a **MEDIUM** score would ask the user to confirm first, and a **LOW** score would be ignored. The cut-offs are chosen on the validation set:

* `t_high`: the lowest score at which precision is at least 90% (few false alarms)
* `t_med`: the highest score at which recall is still at least 90% (few missed screams)

A missed scream is worse than a false alarm, which is why the MEDIUM band exists: it catches screams the model is unsure about without alerting contacts every time.

In [ ]:
# ---- 8. thresholds and bands ----
def choose_thresholds(y_true, p, target_precision=0.90, target_recall=0.90):
    prec, rec, thr = precision_recall_curve(y_true, p)
    prec, rec = prec[:-1], rec[:-1]
    ok_p = np.where(prec >= target_precision)[0]
    ok_r = np.where(rec >= target_recall)[0]
    t_high = float(thr[ok_p].min()) if len(ok_p) else 0.90
    t_med = float(thr[ok_r].max()) if len(ok_r) else 0.30
    return min(t_med, t_high), t_high

t_med, t_high = choose_thresholds(y[va], p_val)
band = lambda p: 'HIGH' if p >= t_high else 'MEDIUM' if p >= t_med else 'LOW'
print(f't_med = {t_med:.3f}   t_high = {t_high:.3f}')
if t_high - t_med < 1e-6:
    print('Note: the two thresholds coincide on this data, so the MEDIUM band is empty. Lower target_precision or raise target_recall, or say so in the report.')

bands = pd.Series([band(p) for p in p_te], name='band')
truth = pd.Series(np.where(y[te] == 1, 'real scream', 'not scream'), name='true class')
table = pd.crosstab(truth, bands).reindex(columns=['HIGH', 'MEDIUM', 'LOW'], fill_value=0)
print('\nTest clips by band (counts):'); print(table)
print('\nShare of each true class falling in each band:'); print((table.div(table.sum(1), axis=0) * 100).round(1))

## 9. False-positive analysis

Which everyday sounds does the model wrongly flag? This uses the ESC-50 validation and test folds (clips the model never trained on), grouped by sound category. A model that screams "danger" at a laughing group or a crying baby will annoy users, so these are the cases worth discussing in the report.

In [ ]:
# ---- 9. false positives by category ----
vt = (va | te) & (y == 0) & (meta['category'] != 'kaggle_nonscream').values
p_vt = clf.predict_proba(X[vt])[:, 1]
fp = (pd.DataFrame({'category': meta.loc[vt, 'category'].values, 'flagged': p_vt >= t_med})
      .groupby('category')['flagged'].mean().sort_values(ascending=False) * 100)
print(fp.head(10).round(1).to_string())
fp.head(10).iloc[::-1].plot.barh(color='#D4235A')
plt.xlabel('% of clips flagged MEDIUM or HIGH'); plt.title('Everyday sounds most often mistaken for a scream')
plt.tight_layout(); plt.show()

## 10. Save the model and a prediction function

In [ ]:
# ---- 10. save and predict ----
joblib.dump({'model': clf, 't_med': t_med, 't_high': t_high, 'sr': SR, 'max_seconds': MAX_SECONDS},
            'distress_model.joblib')

def predict_clip(path):
    f, _ = extract(path)
    p = float(clf.predict_proba(f.reshape(1, -1))[0, 1])
    return {'distress_probability': round(p, 3), 'band': band(p)}

try:
    from google.colab import files
    files.download('distress_model.joblib')
except ImportError:
    pass
# example: predict_clip('kaggle_scream/.../some_clip.wav')

## Limitations to state in the report

* **Not real emergencies.** The clips are public recordings and film or game audio, so real-world performance will be lower. Phone microphones, distance, wind and background noise are not covered.
* **Binary and scream-only.** Crying, shouting for help and other distress sounds are future classes.
* **Small, random splits for the scream clips.** If several clips come from one source recording, the test score can be optimistic. ESC-50's folds protect the negatives but not necessarily the positives.
* **Server-side inference (planned).** The audio would be sent to a small Python service called by the Spring Boot backend. That service is not built yet.
* **Privacy.** Continuously analysing a person's audio is sensitive. A real deployment needs explicit consent and a plan for where audio is processed and stored.

## Related work to cite

* Scream detection with machine learning on balanced AudioSet data, validated on separate real-world audio (JMIR Formative Research, 2020; PMC7327591)
* YAMNet and AudioSet (Google)
* ESC-50 (Piczak, 2015)
* The Kaggle Human Screaming Detection Dataset (check its license and author before citing)